# Day 9: Cohort & Retention Analysis
## Early Warning System for Academic Probation Using Predictive Analytics

This notebook compares student cohorts using the available academic snapshot. Since the dataset does not contain repeated enrollment/activity records across multiple periods, results are **cohort comparisons, not measured longitudinal retention**.

## 1. Load libraries and dataset
Run this notebook from the repository root or from the `Day_9_Cohort_Retention` folder.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

candidates = [
    Path('../Dataset/Cleaned_Academic_Probation_Project.csv'),
    Path('Dataset/Cleaned_Academic_Probation_Project.csv'),
    Path('../Dataset/cleaned_academic_probation_project.csv'),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find cleaned dataset. Check the Dataset folder and CSV filename.')

df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
display(df.head())

## 2. Identify fields
The code detects common column-name variations to reduce setup work.

In [ ]:
def find_col(possible_names):
    lookup = {str(c).strip().lower().replace(' ', '_'): c for c in df.columns}
    for name in possible_names:
        key = name.strip().lower().replace(' ', '_')
        if key in lookup:
            return lookup[key]
    return None

cohort_col = find_col(['Admission_Year', 'Admission Year', 'AdmissionYear'])
gpa_col = find_col(['GPA', 'CGPA', 'Average_GPA'])
attendance_col = find_col(['Attendance_Percentage', 'Attendance Percentage', 'Attendance'])
probation_col = find_col(['Probation_Status', 'Probation Status'])
department_col = find_col(['Department', 'Department_Name', 'Branch'])
semester_col = find_col(['Semester', 'Current_Semester'])
lms_col = find_col(['LMS_Login_Count', 'LMS Login Count', 'LMS_Logins'])

required = {'Admission year': cohort_col, 'GPA': gpa_col, 'Attendance': attendance_col, 'Probation status': probation_col}
missing = [label for label, col in required.items() if col is None]
if missing:
    raise ValueError(f'Missing required columns: {missing}. Available columns: {list(df.columns)}')
print('Detected columns:', required)
print('Optional columns:', {'Department': department_col, 'Semester': semester_col, 'LMS logins': lms_col})

## 3. Clean fields and summarize cohorts
Each row is treated as one student record in the available snapshot.

In [ ]:
work = df.copy()
work['_cohort_year'] = pd.to_numeric(work[cohort_col], errors='coerce')
work['_gpa'] = pd.to_numeric(work[gpa_col], errors='coerce')
work['_attendance'] = pd.to_numeric(work[attendance_col], errors='coerce')
work['_probation'] = work[probation_col].astype(str).str.strip().str.lower().map({'yes': 1, 'y': 1, 'true': 1, '1': 1, 'no': 0, 'n': 0, 'false': 0, '0': 0})
if lms_col:
    work['_lms_logins'] = pd.to_numeric(work[lms_col], errors='coerce')

grouped = work.dropna(subset=['_cohort_year']).groupby('_cohort_year').agg(
    student_records=('_cohort_year', 'size'),
    average_gpa=('_gpa', 'mean'),
    average_attendance_pct=('_attendance', 'mean'),
    probation_records=('_probation', 'sum'),
    known_probation_records=('_probation', 'count')
).reset_index().rename(columns={'_cohort_year': 'admission_year'})
grouped['probation_rate_pct'] = np.where(grouped['known_probation_records'] > 0, 100 * grouped['probation_records'] / grouped['known_probation_records'], np.nan)
if lms_col:
    lms_summary = work.dropna(subset=['_cohort_year']).groupby('_cohort_year')['_lms_logins'].mean().rename('average_lms_login_count').reset_index().rename(columns={'_cohort_year':'admission_year'})
    grouped = grouped.merge(lms_summary, on='admission_year', how='left')
grouped = grouped.sort_values('admission_year')
display(grouped.round(2))

## 4. Optional breakdowns
If Department or Semester is available, summarize the current snapshot by that field too.

In [ ]:
def category_summary(col, label):
    if col is None:
        print(f'{label} field not available; skipping this breakdown.')
        return None
    temp = work.copy()
    result = temp.groupby(col, dropna=False).agg(
        student_records=(col, 'size'),
        average_gpa=('_gpa', 'mean'),
        average_attendance_pct=('_attendance', 'mean'),
        probation_records=('_probation', 'sum'),
        known_probation_records=('_probation', 'count')
    ).reset_index()
    result['probation_rate_pct'] = np.where(result['known_probation_records'] > 0, 100 * result['probation_records'] / result['known_probation_records'], np.nan)
    print(f'{label} summary')
    display(result.round(2))
    return result

department_summary = category_summary(department_col, 'Department')
semester_summary = category_summary(semester_col, 'Semester')

## 5. Visualize cohort comparisons
Charts show differences between admission-year groups; they do not establish that students stayed enrolled over time.

In [ ]:
output_dir = Path('Day_9_Cohort_Retention/Output_Charts') if Path('Day_9_Cohort_Retention').exists() else Path('Output_Charts')
output_dir.mkdir(parents=True, exist_ok=True)
if grouped.empty:
    raise ValueError('No valid admission years were found to compare.')

plt.figure(figsize=(9, 5))
plt.bar(grouped['admission_year'].astype(int).astype(str), grouped['student_records'])
plt.title('Student Records by Admission-Year Cohort')
plt.xlabel('Admission year'); plt.ylabel('Student records'); plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(output_dir / 'student_records_by_cohort.png', dpi=150); plt.show()

plt.figure(figsize=(9, 5))
plt.bar(grouped['admission_year'].astype(int).astype(str), grouped['average_gpa'])
plt.title('Average GPA by Admission-Year Cohort')
plt.xlabel('Admission year'); plt.ylabel('Average GPA'); plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(output_dir / 'average_gpa_by_cohort.png', dpi=150); plt.show()

plt.figure(figsize=(9, 5))
plt.bar(grouped['admission_year'].astype(int).astype(str), grouped['probation_rate_pct'])
plt.title('Recorded Probation Rate by Admission-Year Cohort')
plt.xlabel('Admission year'); plt.ylabel('Probation rate (%)'); plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(output_dir / 'probation_rate_by_cohort.png', dpi=150); plt.show()
print('Charts saved in:', output_dir)

## 6. Export summaries
CSV files are created for use in the report or dashboard.

In [ ]:
grouped.to_csv(output_dir / 'cohort_summary.csv', index=False)
if department_summary is not None:
    department_summary.to_csv(output_dir / 'department_snapshot_summary.csv', index=False)
if semester_summary is not None:
    semester_summary.to_csv(output_dir / 'semester_snapshot_summary.csv', index=False)
print('Summary CSV files saved in:', output_dir)
display(grouped.round(2))

## 7. Interpretation and limitation
Use the tables to identify cohorts with comparatively lower GPA, attendance, or higher recorded probation rates. These are descriptive differences, not proof that admission year caused the outcomes.

**Retention limitation:** this dataset contains a current student snapshot, not repeated enrollment status or activity observations over time. Therefore this notebook does not calculate true retention, dropout, or semester-to-semester persistence. Those measures require longitudinal records with student IDs and period-specific enrollment/activity status.